# AN-RA Cluster Worker
Run all cells to join the distributed training cluster.

**Prerequisites:**
1. Dashboard is deployed and running
2. You have the coordinator URL from the dashboard
3. The shared Drive folder is shared with this Gmail account as **Editor** and added as a My Drive shortcut
4. An-Ra codebase is on this account's Drive at the path below

In [ ]:
# Cell 1: Install dependencies
!pip install -q fastapi requests torch numpy --quiet

# Clone the cluster worker code
import os
if not os.path.exists('/content/anra_worker'):
    !git clone https://github.com/dhurv0045com-spec/gpu-cluster-gmail.git /content/anra_worker 2>&1 | tail -1

import sys
sys.path.insert(0, '/content/anra_worker/worker')

In [ ]:
# Cell 2: Mount Google Drive
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')

# Shared-with-me folders are not mounted directly by Colab. A shortcut in
# My Drive is the reliable path; Shared drives are checked as a fallback.
cluster_candidates = [
    Path('/content/drive/MyDrive/AnRa/cluster'),
    Path('/content/drive/Shareddrives/AnRa/cluster'),
]
DETECTED_CLUSTER_FOLDER = next((str(p) for p in cluster_candidates if p.exists()), None)
if DETECTED_CLUSTER_FOLDER:
    print(f'Cluster folder detected: {DETECTED_CLUSTER_FOLDER}')
else:
    print('Cluster folder is not mounted. In Drive > Shared with me, right-click the shared cluster folder,')
    print('choose Organize > Add shortcut, place it at MyDrive/AnRa/cluster, then remount Drive.')

In [ ]:
# Cell 3: Configuration — fill these in from the dashboard
COORDINATOR_URL = "https://your-app.railway.app"
WORKER_ID = "worker_A"
ACCOUNT_EMAIL = "your_email@gmail.com"
ANRA_REPO_PATH = "/content/drive/MyDrive/AnRa/v2"
CHECKPOINT_PATH = "/content/drive/MyDrive/AnRa/v2/checkpoints/anra_frontier_500m.pt"
TRAINING_DATA = "/content/drive/MyDrive/AnRa/v2/training_data/anra_training.txt"
CLUSTER_DRIVE_FOLDER = DETECTED_CLUSTER_FOLDER or "/content/drive/MyDrive/AnRa/cluster"
if not Path(CLUSTER_DRIVE_FOLDER).exists():
    raise FileNotFoundError(f'Cluster folder unavailable: {CLUSTER_DRIVE_FOLDER}. Follow the shortcut guidance above.')

print("Configuration loaded.")
print(f"Coordinator: {COORDINATOR_URL}")
print(f"Worker ID: {WORKER_ID}")

In [ ]:
# Cell 4: Register with coordinator
import requests
resp = requests.post(f"{COORDINATOR_URL}/api/workers/register", json={
    "worker_id": WORKER_ID,
    "account_email": ACCOUNT_EMAIL,
    "drive_folder_id": CLUSTER_DRIVE_FOLDER
}, timeout=30)
resp.raise_for_status()
assignment = resp.json()
print(f"Registered as slot {assignment['assigned_slot']}")
print(f"Master weights at: {assignment['master_weights_path']}")

In [ ]:
# Cell 5: Launch training loop
import sys
sys.path.insert(0, ANRA_REPO_PATH)
sys.path.insert(0, '/content/anra_worker/worker')

from worker_loop import run_worker_loop

# Import An-Ra model classes
try:
    from anra_brain import CausalTransformerV2
    from tokenizer.tokenizer import AnRaTokenizer
    model_class = CausalTransformerV2
    tokenizer_class = AnRaTokenizer
    model_kwargs = {}
    print("Using An-Ra CausalTransformerV2")
except ImportError as e:
    print(f"WARNING: Could not import An-Ra modules ({e}). Using fallback for testing.")
    import torch.nn as nn
    class DummyModel(nn.Module):
        def __init__(self, **kwargs):
            super().__init__()
            self.embed = nn.Embedding(50257, 1024)
            self.layers = nn.Sequential(
                nn.Linear(1024, 4096), nn.GELU(),
                nn.Linear(4096, 50257),
            )
        def forward(self, x):
            h = self.embed(x)
            return self.layers(h), None
    model_class = DummyModel
    tokenizer_class = None
    model_kwargs = {}

print("Starting worker loop...")
run_worker_loop(
    coordinator_url=COORDINATOR_URL,
    worker_id=WORKER_ID,
    account_email=ACCOUNT_EMAIL,
    anra_repo_path=ANRA_REPO_PATH,
    checkpoint_path=CHECKPOINT_PATH,
    training_data=TRAINING_DATA,
    cluster_drive_folder=CLUSTER_DRIVE_FOLDER,
    model_class=model_class,
    tokenizer_class=tokenizer_class,
    model_kwargs=model_kwargs,
)